In [2]:
import pandas as pd

# 1. Load data
print("Membaca data...")
df = pd.read_csv('online_retail_II.csv.')
initial_rows = len(df)

# 2. Hapus baris yang Customer ID-nya kosong
# Menggunakan .copy() adalah best practice agar terhindar dari SettingWithCopyWarning
df_clean = df.dropna(subset=['Customer ID']).copy()

# 3. Hapus transaksi yang dibatalkan (Invoice diawali huruf 'C')
df_clean = df_clean[~df_clean['Invoice'].astype(str).str.startswith('C')]

# 4. Hapus kuantitas atau harga yang tidak masuk akal (<= 0)
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['Price'] > 0)]

# 5. Merapikan tipe data
# Ubah InvoiceDate jadi tipe datetime bawaan pandas
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
# Ubah Customer ID jadi integer (menghilangkan .0), lalu jadikan string/teks
df_clean['Customer ID'] = df_clean['Customer ID'].astype(int).astype(str)

# 6. Feature Engineering: Bikin kolom Total Pendapatan
df_clean['TotalSales'] = df_clean['Quantity'] * df_clean['Price']

# --- REPORTING ---
final_rows = len(df_clean)
print("\n=== Laporan Data Cleansing ===")
print(f"Jumlah baris awal         : {initial_rows:,}")
print(f"Jumlah baris bersih       : {final_rows:,}")
print(f"Total baris yang dibuang  : {initial_rows - final_rows:,}")
print("\n=== Sampel Data Bersih ===")
print(df_clean.head())

Membaca data...

=== Laporan Data Cleansing ===
Jumlah baris awal         : 1,067,371
Jumlah baris bersih       : 805,549
Total baris yang dibuang  : 261,822

=== Sampel Data Bersih ===
  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

          InvoiceDate  Price Customer ID         Country  TotalSales  
0 2009-12-01 07:45:00   6.95       13085  United Kingdom        83.4  
1 2009-12-01 07:45:00   6.75       13085  United Kingdom        81.0  
2 2009-12-01 07:45:00   6.75       13085  United Kingdom        81.0  
3 2009-12-01 07:45:00   2.10       13085  United Kingdom       100.8  
4 2009-12-01 07:45:00   1.25       130

In [3]:
import datetime as dt

# 1. Tentukan "Hari Ini" (Reference Date)
# Biasanya diatur 1 hari setelah tanggal transaksi terakhir yang ada di dataset
latest_date = df_clean['InvoiceDate'].max()
reference_date = latest_date + dt.timedelta(days=1)

# 2. Hitung R, F, dan M untuk masing-masing Customer ID
print("Menghitung metrik RFM...")
rfm = df_clean.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (reference_date - x.max()).days,  # Recency
    'Invoice': 'nunique',                                      # Frequency
    'TotalSales': 'sum'                                        # Monetary
}).reset_index()

# 3. Ganti nama kolom agar sesuai standar bisnis
rfm.rename(columns={
    'InvoiceDate': 'Recency',
    'Invoice': 'Frequency',
    'TotalSales': 'Monetary'
}, inplace=True)

# 4. Tampilkan hasil
print("\n=== Sampel Data RFM ===")
print(rfm.head())

print("\n=== Ringkasan Statistik RFM ===")
print(rfm.describe())

Menghitung metrik RFM...

=== Sampel Data RFM ===
  Customer ID  Recency  Frequency  Monetary
0       12346      326         12  77556.46
1       12347        2          8   5633.32
2       12348       75          5   2019.40
3       12349       19          4   4428.69
4       12350      310          1    334.40

=== Ringkasan Statistik RFM ===
           Recency    Frequency       Monetary
count  5878.000000  5878.000000    5878.000000
mean    201.331916     6.289384    3018.616737
std     209.338707    13.009406   14737.731040
min       1.000000     1.000000       2.950000
25%      26.000000     1.000000     348.762500
50%      96.000000     3.000000     898.915000
75%     380.000000     7.000000    2307.090000
max     739.000000   398.000000  608821.650000


In [4]:
# 1. Bikin skor kuartil 1-4 untuk R, F, dan M
# Untuk Recency, makin kecil (makin baru) angkanya, skornya makin tinggi (4)
rfm['R'] = pd.qcut(rfm['Recency'], q=4, labels=[4, 3, 2, 1])

# Untuk Frequency dan Monetary, makin besar angkanya makin bagus (skor 4)
# Frequency pakai rank(method='first') buat ngakalin data yang nilainya sama
rfm['F'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=4, labels=[1, 2, 3, 4])
rfm['M'] = pd.qcut(rfm['Monetary'], q=4, labels=[1, 2, 3, 4])

# 2. Gabungkan jadi satu skor (contoh: '444', '111')
rfm['RFM_Score'] = rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)

# 3. Klasifikasi Segmen Pelanggan
def categorize(x):
    if x['RFM_Score'] == '444':
        return 'Best Customers'   # Belanja baru-baru ini, sering, nominal gede
    elif x['R'] >= 3 and x['F'] >= 3:
        return 'Loyal Customers'
    elif x['R'] <= 2 and x['F'] >= 3:
        return 'At Risk'          # Dulunya sering, tapi udah lama hilang
    elif x['R'] <= 2 and x['F'] <= 2:
        return 'Lost'             # Udah lama hilang, nominal/frekuensi kecil
    else:
        return 'Regulars'         # Sisanya yang biasa-biasa aja

rfm['Segment'] = rfm.apply(categorize, axis=1)

# Tampilkan jumlah pelanggan per segmen
print("\n=== Jumlah Pelanggan per Segmen ===")
print(rfm['Segment'].value_counts())
print("\n=== Sampel Hasil ===")
print(rfm[['Customer ID', 'RFM_Score', 'Segment']].head())


=== Jumlah Pelanggan per Segmen ===
Segment
Lost               2045
Loyal Customers    1398
Regulars            894
At Risk             880
Best Customers      661
Name: count, dtype: int64

=== Sampel Hasil ===
  Customer ID RFM_Score          Segment
0       12346       244          At Risk
1       12347       444   Best Customers
2       12348       333  Loyal Customers
3       12349       434  Loyal Customers
4       12350       211             Lost


In [5]:
# Ekspor data hasil segmentasi RFM
rfm.to_csv('rfm_segmentation_result.csv', index=False)
print("Data segmentasi RFM berhasil diekspor dan siap divisualisasikan!")

Data segmentasi RFM berhasil diekspor dan siap divisualisasikan!
